# Notebook 03 – Geomarketing-Scoring-Modell & Top-5-Standorte
**Charge & Ride Züri** | ZHAW Geomarketing FS2026

Dieses Notebook implementiert das Scoring-Modell zur Identifikation
der 5 optimalen Standorte für neue E-Bike-Ladestationen.

**Voraussetzungen:** Notebooks 01 und 02 müssen vollständig ausgeführt worden sein.

**Methodik:**
1. Supply-Analyse: 3 km-Puffer um bestehende Ladestationen
2. Demand-Analyse: KDE-Heatmap der Velofrequenzen
3. Scoring: POIs ausserhalb Versorgungszone bewerten
4. Greedy-Selektion: Top-5 iterativ wählen – jeder neu gewählte Standort
   fliesst als zusätzliche Ladesäule in die nächste Runde ein, sodass
   räumliche Konzentration automatisch vermieden wird.

**Outputs:**
- `supply_zone_3km.geojson` – Versorgungszone bestehender Ladestationen
- `alle_kandidaten_scored.geojson` – Alle bewerteten Kandidaten-POIs
- `top5_standorte.geojson` – Die 5 Topstandorte (für QGIS-Karte 3)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
import contextily as cx
import folium
from folium.plugins import HeatMap

ROOT = Path('..').resolve()
DATA_PROCESSED = ROOT / 'data' / 'processed'

# Scoring-Gewichtungen (anpassbar – Begruendung im Bericht!)
W_FREQUENZ = 0.5    # Gewicht: Velofrequenz
W_DISTANZ = 0.5     # Gewicht: Distanz zur naechsten Ladesaeule

BUFFER_RADIUS_M = 3000  # 3 km Versorgungszone

print(f'Scoring-Gewichtungen: Frequenz={W_FREQUENZ}, Distanz={W_DISTANZ}')

## 1. Daten laden (aus Notebook 01 und 02)

In [ ]:
# Alle verarbeiteten Daten laden
gdf_ladestationen = gpd.read_file(DATA_PROCESSED / 'ladestationen_zh.geojson').to_crs(epsg=2056)
gdf_gastro        = gpd.read_file(DATA_PROCESSED / 'gastronomie_zh.geojson').to_crs(epsg=2056)
gdf_bahnhoefe     = gpd.read_file(DATA_PROCESSED / 'bahnhoefe_zh.geojson').to_crs(epsg=2056)
gdf_kanton        = gpd.read_file(DATA_PROCESSED / 'kanton_zh_boundary.geojson').to_crs(epsg=2056)

# Velozählstellen: Stadt ZH (25) + Kanton ZH (97) kombinieren
gdf_stadt = gpd.read_file(DATA_PROCESSED / 'velozaehlstellen_stadt_zh.geojson').to_crs(epsg=2056)
gdf_kanton_zh = gpd.read_file(DATA_PROCESSED / 'velozaehlstellen_kanton_zh.geojson').to_crs(epsg=2056)

# Gemeinsame Spalten für KDE: geometry + dtv_velos
gdf_zaehlstellen = pd.concat([
    gdf_stadt[['geometry', 'dtv_velos']],
    gdf_kanton_zh[['geometry', 'dtv_velos']]
], ignore_index=True)
gdf_zaehlstellen = gpd.GeoDataFrame(gdf_zaehlstellen, crs='EPSG:2056')

print('Daten geladen:')
print(f'  Ladestationen:           {len(gdf_ladestationen)}')
print(f'  Gastronomie:             {len(gdf_gastro)}')
print(f'  Bahnhoefe:               {len(gdf_bahnhoefe)}')
print(f'  Zaehlstellen gesamt:     {len(gdf_zaehlstellen)} (Stadt: {len(gdf_stadt)}, Kanton: {len(gdf_kanton_zh)})')
print(f'  Zaehlstellen mit DTV:    {gdf_zaehlstellen["dtv_velos"].notna().sum()}')

## 2. Supply-Analyse: Versorgungszone (3 km-Puffer)

In [ ]:
# 3 km-Puffer um bestehende Ladestationen (in LV95: Meter!)
puffer_einzeln = gdf_ladestationen.geometry.buffer(BUFFER_RADIUS_M)

# Alle Puffer zu einem einzigen Polygon vereinen
from shapely.ops import unary_union
supply_zone = unary_union(puffer_einzeln)

gdf_supply = gpd.GeoDataFrame({'geometry': [supply_zone]}, crs='EPSG:2056')

# Speichern
gdf_supply.to_crs(epsg=4326).to_file(DATA_PROCESSED / 'supply_zone_3km.geojson', driver='GeoJSON')

# Versorgungsquote berechnen
kanton_area = gdf_kanton.geometry.area.sum()
supply_area = supply_zone.area
print(f'Versorgungsquote Kanton ZH: {supply_area/kanton_area*100:.1f}%')
print('Supply-Zone gespeichert.')

## 3. Kandidaten-POIs zusammenstellen und filtern

In [ ]:
# Kandidaten = Gastronomie + Bahnhöfe
gdf_gastro['poi_typ'] = 'gastronomie'
gdf_bahnhoefe['poi_typ'] = 'bahnhof'

# Gemeinsamen GeoDataFrame erstellen
cols_needed = ['geometry', 'name', 'poi_typ']
gdf_kandidaten = pd.concat([
    gdf_gastro[[c for c in cols_needed if c in gdf_gastro.columns]],
    gdf_bahnhoefe[[c for c in cols_needed if c in gdf_bahnhoefe.columns]]
], ignore_index=True)
gdf_kandidaten = gpd.GeoDataFrame(gdf_kandidaten, crs='EPSG:2056')

# Nur POIs AUSSERHALB der Versorgungszone behalten
gdf_kandidaten['in_supply_zone'] = gdf_kandidaten.geometry.within(supply_zone)
gdf_weisse_flecken = gdf_kandidaten[~gdf_kandidaten['in_supply_zone']].copy()

print(f'Alle Kandidaten:            {len(gdf_kandidaten)}')
print(f'Ausserhalb Versorgungszone: {len(gdf_weisse_flecken)}')

## 4. Scoring-Kriterium 1: Distanz zur nächsten Ladesäule

In [ ]:
from shapely.ops import nearest_points

# Distanz zur naechsten bestehenden Ladesaeule berechnen
ladesaeule_union = unary_union(gdf_ladestationen.geometry)

gdf_weisse_flecken['distanz_ladesaeule_m'] = gdf_weisse_flecken.geometry.apply(
    lambda p: p.distance(ladesaeule_union)
)

print(f'Max. Distanz: {gdf_weisse_flecken["distanz_ladesaeule_m"].max():.0f} m')
print(f'Mittlere Distanz: {gdf_weisse_flecken["distanz_ladesaeule_m"].mean():.0f} m')

## 5. Scoring-Kriterium 2: Nähe zu Velo-Hotspots (KDE)

In [ ]:
# Kernel Density Estimation (KDE) der Velozählstellen-Frequenzen
# Gewichtet nach mittlerer Tagesfrequenz (dtv_velos)

# Nur Zählstellen mit gültigem DTV-Wert verwenden
gdf_kde = gdf_zaehlstellen[gdf_zaehlstellen['dtv_velos'].notna()].copy()

# Koordinaten der Zählstellen
coords = np.array([
    [p.x for p in gdf_kde.geometry],
    [p.y for p in gdf_kde.geometry]
])

# KDE gewichtet nach DTV
weights = gdf_kde['dtv_velos'].values
kde = gaussian_kde(coords, weights=weights)

# KDE-Wert fuer jeden Kandidaten berechnen
kandidaten_coords = np.array([
    [p.x for p in gdf_weisse_flecken.geometry],
    [p.y for p in gdf_weisse_flecken.geometry]
])

gdf_weisse_flecken['kde_velo_score'] = kde(kandidaten_coords)
print(f'KDE-Score berechnet. Basis: {len(gdf_kde)} Zählstellen')

## 6. Gesamt-Score berechnen und Top-5 ausgeben

In [ ]:
# Min-Max-Normalisierung (0-1)
def normalize(series):
    """Min-Max-Normalisierung auf [0, 1]"""
    min_val = series.min()
    max_val = series.max()
    if max_val == min_val:
        return series * 0
    return (series - min_val) / (max_val - min_val)

# KDE-Score einmalig global normalisieren (aendert sich nicht)
gdf_weisse_flecken['norm_kde'] = normalize(gdf_weisse_flecken['kde_velo_score'])


def greedy_reselect(gdf_kandidaten, gdf_ladestationen_initial, n=5, puffer_m=BUFFER_RADIUS_M):
    """
    Greedy-Selektion mit iterativer Neuberechnung der Distanzscores.

    Nach jeder Auswahl wird der gewaeehlte Standort zu den bestehenden
    Ladestationen hinzugefuegt. Zudem werden alle Kandidaten innerhalb
    des Versorgungspuffers (puffer_m) des neuen Standorts aus dem Pool
    entfernt – analog zur initialen Filterung der weissen Flecken.

    Ablauf pro Runde:
      1. Distanz zur naechsten Ladesaeule neu berechnen (inkl. bereits gewaeehlter)
      2. Distanz- und Gesamt-Score neu normalisieren und berechnen
      3. Besten Kandidaten waehlen
      4. Kandidaten innerhalb des Puffers aus dem Pool entfernen
    """
    ausgewaehlte = []
    verbleibend = gdf_kandidaten.copy()
    aktuelle_ladesaeule_geom = unary_union(gdf_ladestationen_initial.geometry)

    for runde in range(n):
        if verbleibend.empty:
            break

        # Distanz zur naechsten Ladesaeule neu berechnen
        verbleibend['distanz_ladesaeule_m'] = verbleibend.geometry.apply(
            lambda p: p.distance(aktuelle_ladesaeule_geom)
        )

        # Distanz-Score neu normalisieren (relativ zu verbleibenden Kandidaten)
        verbleibend['norm_distanz'] = normalize(verbleibend['distanz_ladesaeule_m'])

        # Gesamt-Score berechnen
        verbleibend['gesamt_score'] = (
            W_FREQUENZ * verbleibend['norm_kde'] +
            W_DISTANZ  * verbleibend['norm_distanz']
        )

        # Besten Kandidaten waehlen
        best_idx = verbleibend['gesamt_score'].idxmax()
        gewaehlter = verbleibend.loc[best_idx].copy()
        gewaehlter['rang'] = runde + 1
        ausgewaehlte.append(gewaehlter)

        print(f"Rang {runde + 1}: {str(gewaehlter.get('name', 'k.A.')):<35} "
              f"| Score: {gewaehlter['gesamt_score']:.3f} "
              f"| Distanz: {gewaehlter['distanz_ladesaeule_m']/1000:.1f} km")

        # Neu gewaeehlten Standort zur Ladesaeulen-Menge hinzufuegen
        aktuelle_ladesaeule_geom = aktuelle_ladesaeule_geom.union(gewaehlter.geometry)

        # Alle Kandidaten innerhalb des Versorgungspuffers aus dem Pool entfernen
        verbleibend = verbleibend.drop(index=best_idx)
        zu_nah = verbleibend.geometry.distance(gewaehlter.geometry) < puffer_m
        n_entfernt = zu_nah.sum()
        verbleibend = verbleibend[~zu_nah]
        print(f"         -> {n_entfernt} Kandidaten im {puffer_m/1000:.0f}-km-Puffer entfernt "
              f"({len(verbleibend)} verbleibend)")

    result = gpd.GeoDataFrame(ausgewaehlte, crs=gdf_kandidaten.crs).reset_index(drop=True)
    return result


gdf_top5 = greedy_reselect(gdf_weisse_flecken, gdf_ladestationen, n=5)

print('\nTOP 5 STANDORTE (Greedy-Selektion mit Puffer-Exklusion):')
print(gdf_top5[['rang', 'name', 'poi_typ', 'gesamt_score', 'distanz_ladesaeule_m']]
      .to_string(index=False))

# Paerweise Abstände zur Verifikation
from itertools import combinations
print('\nPaarweise Abstaende zwischen Top-5-Standorten:')
for (_, a), (_, b) in combinations(gdf_top5.iterrows(), 2):
    d = a.geometry.distance(b.geometry)
    print(f'  Rang {a["rang"]} <-> Rang {b["rang"]}: {d/1000:.1f} km')


## 7. Ergebnisse speichern

In [ ]:
# Alle Kandidaten speichern
gdf_weisse_flecken.to_crs(epsg=4326).to_file(
    DATA_PROCESSED / 'alle_kandidaten_scored.geojson', driver='GeoJSON'
)

# Top 5 speichern
gdf_top5.to_crs(epsg=4326).to_file(
    DATA_PROCESSED / 'top5_standorte.geojson', driver='GeoJSON'
)

print('Ergebnisse gespeichert.')
print('-> Naechster Schritt: GeoJSONs in QGIS laden und Karten layouten!')


## 8. Interaktive Karte (Folium)

In [ ]:
# Interaktive Folium-Karte mit allen Ergebnissen
center = [47.4, 8.65]  # Zentrum Kanton Zuerich
m = folium.Map(location=center, zoom_start=10, tiles='OpenStreetMap')

# Heatmap der Velofrequenzen
heat_data = [
    [row.geometry.y, row.geometry.x, row['dtv_velos']]
    for _, row in gdf_zaehlstellen.to_crs(epsg=4326).iterrows()
    if not pd.isna(row.get('dtv_velos', np.nan))
]
HeatMap(heat_data, name='Velo-Hotspots', min_opacity=0.3).add_to(m)

# Bestehende Ladestationen
for _, row in gdf_ladestationen.to_crs(epsg=4326).iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5, color='green', fill=True,
        popup=f"Bestehend: {row.get('name', 'k.A.')}"
    ).add_to(m)

# Top-5-Standorte (Rang aus MMR-Selektion)
for _, row in gdf_top5.to_crs(epsg=4326).iterrows():
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        popup=f"TOP {row['rang']}: {row.get('name', 'k.A.')} | Score: {row['gesamt_score']:.3f}",
        icon=folium.Icon(color='red', icon='bolt', prefix='fa')
    ).add_to(m)

folium.LayerControl().add_to(m)

# Speichern
m.save(str(DATA_PROCESSED / 'interaktive_karte.html'))
print('Interaktive Karte gespeichert: data/processed/interaktive_karte.html')
m